# Attaching Timestamps to Topic Assignments
<!-- structured-notebook -->

### This step was needed due to missing timestamp collection in the initial runs.

## Notebook Summary
Purpose: recover per-document UTC timestamps after BERTopic assignment so topic outputs can feed temporal analyses (trends, COVID windows, diffusion lags).

Main steps:
- Stream the raw JSONL back in and apply the **exact same** preprocessing that produced the trained model's inputs.
- Build a dictionary mapping `cleaned_text → deque[timestamps]` (deque for deterministic FIFO when duplicates exist).
- Walk `topic_doc_map` and pop timestamps to produce `topic_doc_map_with_ts.pkl`.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/merged_submissions.jsonl` | Produced by `01_data_collection/01_reddit_data_extraction.ipynb` |
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/kept_metadata.parquet` | Produced by `01_preprocessing_and_topic_modelling.ipynb` |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/topic_doc_map_with_ts.pkl` | Consumed by temporal analyses (RQ2, diffusion) |


# Reproducibility Requirement
<!-- structured-notebook -->
## Summary
The core challenge is **exact reproduction of preprocessing**. Any discrepancy (different regex, different spaCy version, different whitespace normalization) will cause cleaned-text mismatches and dropped timestamps.

`single_preprocess_fn` is the canonical single-input function. It reproduces every step of the batch pipeline:

1. URL removal
2. Emoji removal
3. Reddit-syntax removal (`r/`, `u/`, blockquotes)
4. Character filtering (v2: preserves digits, hyphens, `+`, etc.)
5. Lowercasing
6. Language detection (English only, `DetectorFactory.seed = 0`)
7. Minimum token count (`>3`)
8. spaCy lemmatization with stopword / punctuation filtering
9. Whitespace normalization

If the batch pipeline ever changes, update this function in the same commit.


## 1. Setup & Imports

In [ ]:
import sys
import os
import re
import gc
import pickle
from pathlib import Path
from collections import defaultdict, deque
from typing import Callable, Dict, List, Optional

import pandas as pd
import emoji
import spacy
from langdetect import detect, DetectorFactory
from tqdm import tqdm

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from src.shared_reddit_telegram.text_cleaning import clean_text, is_english, lemmatize_texts, process_docs, dedupe_strings, preprocess_pipeline
from src.shared_reddit_telegram.config import PROJECT_ROOT, REDDIT_DATA, REDDIT_SMALL_DATA

# Deterministic language detection
DetectorFactory.seed = 0

# Load spaCy once (same model used during preprocessing)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print(f"Project root: {PROJECT_ROOT}")

## 2. Configuration

In [ ]:
TOPIC_DOC_MAP_PATH = "../scripts/topic_modelling_v2/round_11/topic_doc_map.pkl"
RAW_PATH = str(REDDIT_SMALL_DATA / "merged_submissions.jsonl")
OUT_PATH = "../scripts/topic_modelling_v2/round_11/topic_doc_map_with_ts.pkl"

print(f"Topic-doc map: {TOPIC_DOC_MAP_PATH}")
print(f"Raw data:      {RAW_PATH}")
print(f"Output:        {OUT_PATH}")

## 3. The Matching Strategy

The core challenge is **reproducibility**: the preprocessing must produce *exactly* the same string from a given raw post as the batch pipeline did during topic modeling. Any discrepancy (different regex, different spaCy version, different whitespace normalization) will cause a mismatch.

We define `single_preprocess_fn` as a self-contained function that reproduces every step of the batch pipeline on a single text input:

1. URL removal
2. Emoji removal
3. Reddit syntax removal (`r/`, `u/`, blockquotes)
4. Character filtering (v2: preserves digits, hyphens, `+`, etc.)
5. Lowercasing
6. Language detection (English only)
7. Minimum token count (>3)
8. spaCy lemmatization with stopword/punctuation filtering
9. Whitespace normalization

**Why deque?** When multiple raw posts produce the same preprocessed string, we need a deterministic ordering for timestamp assignment. Using a sorted deque (oldest timestamp first) ensures consistent behavior.

## 4. Define Single-Row Preprocessor

This function must be an exact mirror of the batch preprocessing pipeline. It is intentionally defined inline (not imported from `src/shared_reddit_telegram/`) because it must match the specific preprocessing version used when the topic model was trained.

In [ ]:
def single_preprocess_fn(text: str) -> Optional[str]:
    """
    Exact single-row equivalent of the batch preprocess_texts pipeline.
    
    Steps:
      1. URL/emoji/Reddit-tag removal
      2. Character filtering (v2: keep digits, hyphens, +, /, #, apostrophe)
      3. Lowercasing
      4. Language filter (English only via langdetect)
      5. Minimum length (>3 tokens after cleaning)
      6. spaCy lemmatization + stopword/punctuation filtering
      7. Whitespace normalization
    
    Returns:
        Final preprocessed string, or None if filtered out at any stage.
    """
    if not isinstance(text, str) or len(text) < 5:
        return None

    # --- clean_text (v2) ---
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = emoji.replace_emoji(text, replace="")

    # Reddit-specific syntax
    text = re.sub(r"\b[ru]/\w+\b", " ", text)       # r/subreddit, u/username
    text = re.sub(r"(^|\n)>\s.*", " ", text, flags=re.M)  # blockquotes

    # v2 character filter: preserve \w, hyphens, +, /, #, apostrophe
    text = re.sub(r"[^\w\-\+/#']", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.lower().strip()

    if not text:
        return None

    # Language detection
    try:
        if detect(text) != "en":
            return None
    except Exception:
        return None

    # Minimum token count
    if len(text.split()) <= 3:
        return None

    # --- Lemmatization (must match batch pipeline exactly) ---
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_space or t.is_punct or t.is_stop:
            continue
        txt = t.text.lower()
        if not re.search(r"[a-z0-9]", txt):
            continue
        tokens.append(txt)

    if not tokens:
        return None

    out = " ".join(tokens)
    out = re.sub(r"\s+", " ", out).strip()
    return out


# Quick sanity check
test_input = "Check out r/longevity! I've been taking NAD+ and B12 https://example.com"
test_output = single_preprocess_fn(test_input)
print(f"Input:  {test_input}")
print(f"Output: {test_output}")

## 5. Timestamp Attachment Function

This function streams the raw data, preprocesses each row to reproduce the exact cleaned strings, builds a lookup dictionary, then walks through the topic-doc map to attach timestamps.

In [ ]:
def attach_timestamps_to_existing_topic_map(
    topic_doc_map_path: str,
    raw_path: str,
    out_path: str,
    single_preprocess_fn: Callable[[str], Optional[str]],
    ts_col: str = "created_utc",
    title_col: str = "title",
    body_col: str = "selftext",
    chunksize: int = 200_000,
) -> Dict[int, List[dict]]:
    """
    Enrich an existing topic_doc_map (topic -> [preprocessed strings]) with timestamps,
    WITHOUT refitting or re-transforming the topic model.

    It streams the raw dataset, applies the EXACT single-row preprocessor to each
    (title + selftext), builds a dict cleaned_text -> deque[timestamps], then
    attaches timestamps to the preprocessed strings already stored in topic_doc_map.

    Args:
        topic_doc_map_path: Path to existing topic_doc_map.pkl (topic -> list[str]).
        raw_path: Path to raw data file (JSONL/CSV/Parquet) with timestamps.
        out_path: Where to save the enriched mapping (topic -> list[{text, timestamp}]).
        single_preprocess_fn: Function(str) -> Optional[str] reproducing EXACT preprocessing.
        ts_col: Timestamp column name in raw data.
        title_col: Title column name.
        body_col: Body text column name.
        chunksize: Rows per chunk for streaming.

    Returns:
        dict[int, list[{"text": str, "timestamp": pd.Timestamp}]]
    """
    # 1) Load existing topic -> docs mapping
    with open(topic_doc_map_path, "rb") as f:
        topic_doc_map = pickle.load(f)
    print(f"Loaded topic_doc_map with {len(topic_doc_map)} topics.")

    # 2) Stream raw data and build cleaned_text -> deque[timestamps]
    text2ts = defaultdict(deque)

    if raw_path.endswith(".csv"):
        reader = pd.read_csv(raw_path, chunksize=chunksize)
    elif raw_path.endswith(".parquet"):
        reader = [pd.read_parquet(raw_path)]
    elif raw_path.endswith(".json") or raw_path.endswith(".jsonl"):
        reader = pd.read_json(raw_path, lines=True, chunksize=chunksize)
    else:
        raise ValueError("Unsupported file format. Use CSV/Parquet/JSON/JSONL.")

    total_rows = 0
    kept_rows = 0

    for chunk in tqdm(reader, desc="Indexing timestamps by preprocessed text"):
        total_rows += len(chunk)

        # Parse timestamps
        if ts_col not in chunk.columns:
            raise ValueError(f"Timestamp column '{ts_col}' not found in raw data.")
        if ts_col == "created_utc":
            chunk["_ts"] = pd.to_datetime(chunk[ts_col], unit="s", utc=True, errors="coerce")
        else:
            chunk["_ts"] = pd.to_datetime(chunk[ts_col], utc=True, errors="coerce")

        # Prepare text
        for col in (title_col, body_col):
            if col not in chunk.columns:
                raise ValueError(f"Missing text column '{col}'.")
        chunk = chunk.fillna({title_col: "", body_col: ""}).astype({title_col: str, body_col: str})
        texts = (chunk[title_col] + " " + chunk[body_col]).tolist()
        tss = chunk["_ts"].tolist()

        # Preprocess each row and index by cleaned text
        for raw_txt, ts in zip(texts, tss):
            cleaned = single_preprocess_fn(raw_txt)
            if cleaned:
                text2ts[cleaned].append(ts)
                kept_rows += 1

        del chunk, texts, tss
        gc.collect()

    print(f"Raw rows: {total_rows:,} | Kept after preprocess: {kept_rows:,}")
    print(f"Unique cleaned strings indexed: {len(text2ts):,}")

    # Sort timestamps within each key for deterministic ordering
    for key in list(text2ts.keys()):
        if len(text2ts[key]) > 1:
            text2ts[key] = deque(sorted(text2ts[key]))

    # 3) Build new map with timestamps
    topic_doc_map_with_ts = {}
    unmatched = 0
    total_docs = 0
    unmatched_texts = []

    for topic, docs in topic_doc_map.items():
        augmented = []
        for doc in docs:
            total_docs += 1
            doc_norm = re.sub(r"\s+", " ", doc).strip()
            ts = None
            q = text2ts.get(doc_norm)
            if q and len(q) > 0:
                ts = q.popleft()
            else:
                unmatched += 1
                unmatched_texts.append(doc)
            augmented.append({"text": doc, "timestamp": ts})
        topic_doc_map_with_ts[int(topic)] = augmented

    match_rate = 1 - unmatched / max(1, total_docs)
    print(f"\nMatched: {total_docs - unmatched:,}/{total_docs:,} ({match_rate:.2%})")
    print(f"Unmatched: {unmatched:,} ({unmatched/max(1,total_docs):.2%})")

    if match_rate < 0.90:
        print(f"WARNING: Low match rate ({match_rate:.2%}). "
              f"Check preprocessing consistency and raw data coverage.")

    # Save unmatched docs for debugging
    if unmatched_texts:
        diag_path = out_path.replace(".pkl", "_unmatched.txt")
        with open(diag_path, "w") as f:
            for d in unmatched_texts:
                f.write(d + "\n")
        print(f"Saved unmatched diagnostics to: {diag_path}")

    # 4) Save enriched mapping
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "wb") as f:
        pickle.dump(topic_doc_map_with_ts, f)
    print(f"Saved enriched topic-doc map to: {out_path}")

    return topic_doc_map_with_ts

## 6. Run Timestamp Attachment

In [ ]:
topic_doc_map_with_ts = attach_timestamps_to_existing_topic_map(
    topic_doc_map_path=TOPIC_DOC_MAP_PATH,
    raw_path=RAW_PATH,
    out_path=OUT_PATH,
    single_preprocess_fn=single_preprocess_fn,
    ts_col="created_utc",
    title_col="title",
    body_col="selftext",
    chunksize=200_000,
)

## 7. Diagnostics

Inspect the results: match rate, sample of enriched entries, and any unmatched documents.

In [ ]:
# Summary statistics
total_docs = sum(len(docs) for docs in topic_doc_map_with_ts.values())
matched_docs = sum(
    1 for docs in topic_doc_map_with_ts.values()
    for d in docs if d["timestamp"] is not None
)
unmatched_docs = total_docs - matched_docs

print(f"Total documents across all topics: {total_docs:,}")
print(f"Successfully matched with timestamps: {matched_docs:,} ({matched_docs/total_docs:.2%})")
print(f"Unmatched (no timestamp): {unmatched_docs:,} ({unmatched_docs/total_docs:.2%})")
print(f"Number of topics: {len(topic_doc_map_with_ts)}")

In [ ]:
# Show a sample of enriched entries from the first non-outlier topic
sample_topic = min(t for t in topic_doc_map_with_ts.keys() if t >= 0)
print(f"\nSample entries from topic {sample_topic}:")
print("-" * 80)
for entry in topic_doc_map_with_ts[sample_topic][:5]:
    print(f"  Timestamp: {entry['timestamp']}")
    print(f"  Text:      {entry['text'][:120]}...")
    print()

In [ ]:
# Inspect unmatched docs if any exist
diag_path = OUT_PATH.replace(".pkl", "_unmatched.txt")
if os.path.exists(diag_path):
    with open(diag_path, "r") as f:
        unmatched_samples = [line.strip() for line in f.readlines()[:10]]
    print(f"First {len(unmatched_samples)} unmatched documents:")
    for i, doc in enumerate(unmatched_samples):
        print(f"  [{i}] {doc[:150]}")
else:
    print("No unmatched documents -- perfect match rate.")

---
<!-- nav-footer -->

← [01 preprocessing and topic modelling](../../../../../SocialMedia/Reddit/notebooks/BERTopic/02_preprocessing_and_topic_modelling/01_preprocessing_and_topic_modelling.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [01 subtopic analysis](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/01_subtopic_analysis.ipynb) →
